# Settimana 15 — Progetto 2: Agente di Ricerca Autonomo

**Obiettivo:** costruire `ResearchAgent`, un agente che riceve un tema, pianifica la sua ricerca,
cerca fonti sul web, legge e riassume il contenuto, e consegna un report strutturato in Markdown.

**Cosa costruisci:**
- `proyectos/research-agent/agent/tools.py` — strumenti dell'agente
- `proyectos/research-agent/agent/planner.py` — pianificazione con LLM
- `proyectos/research-agent/agent/executor.py` — ciclo ReAct per sottotask
- `proyectos/research-agent/agent/synthesizer.py` — sintesi del report finale
- `proyectos/research-agent/agent/agent.py` — classe `ResearchAgent` che orchestra tutto
- `proyectos/research-agent/app.py` — CLI per lanciare l'agente

**Prerequisiti:** settimane 9 (ReAct), 10 (memoria), 11 (Plan-and-Execute).

**Costo stimato di questo notebook:** ~$0.01–0.05 con `gpt-4o-mini` (3-5 report di prova).

## Cella 0 — Setup

In [ ]:
import subprocess, sys

paquetes = ["openai", "httpx", "beautifulsoup4", "python-dotenv"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + paquetes)

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Definisci OPENAI_API_KEY nel tuo .env"
print("✓ API key caricata")

# Directory del progetto
PROJECT_DIR = Path("../proyectos/research-agent").resolve()
AGENT_DIR   = PROJECT_DIR / "agent"
INFORMES_DIR = PROJECT_DIR / "informes"

for d in [AGENT_DIR, INFORMES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Progetto in: {PROJECT_DIR}")

## Cella 1 — Strumenti dell'agente (`tools.py`)

Implementiamo quattro strumenti. Inizia verificando che `buscar_web` e `leer_url` funzionino
prima di incapsularli nell'agente.

In [ ]:
TOOLS_PY = '''
"""Strumenti per ResearchAgent."""
import httpx
from bs4 import BeautifulSoup


def buscar_web(query: str, max_resultados: int = 5) -> list[dict]:
    """
    Cerca su DuckDuckGo (senza API key) e restituisce una lista di
    {titulo, url, descripcion}.
    """
    params = {
        "q": query,
        "format": "json",
        "no_html": "1",
        "skip_disambig": "1",
    }
    try:
        r = httpx.get("https://api.duckduckgo.com/", params=params, timeout=10)
        data = r.json()
    except Exception as e:
        return [{"titulo": "Error", "url": "", "descripcion": str(e)}]

    resultados = []
    for item in data.get("RelatedTopics", []):
        if "FirstURL" in item and len(resultados) < max_resultados:
            resultados.append({
                "titulo":      item.get("Text", "")[:120],
                "url":         item["FirstURL"],
                "descripcion": item.get("Text", ""),
            })
    return resultados


def leer_url(url: str, max_chars: int = 4000) -> str:
    """
    Scarica una URL e restituisce il testo pulito (fino a max_chars caratteri).
    Elimina script, stili e navigazione.
    """
    if not url or url.startswith("https://duckduckgo.com"):
        return "URL non valida o non accessibile direttamente."
    try:
        r = httpx.get(
            url, timeout=15, follow_redirects=True,
            headers={"User-Agent": "Mozilla/5.0 (compatible; ResearchBot/1.0)"}
        )
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        texto = soup.get_text(separator=" ", strip=True)
        # Comprime spazi multipli
        import re
        texto = re.sub(r"\s+", " ", texto)
        return texto[:max_chars]
    except Exception as e:
        return f"Errore leggendo {url}: {e}"


def buscar_en_corpus(query: str, corpus_path: str = "./chroma_db") -> str:
    """
    Cerca nel corpus locale (Chroma) se esiste.
    Restituisce testo vuoto se non c'è corpus disponibile.
    """
    try:
        import chromadb
        from openai import OpenAI
        import os

        client = chromadb.PersistentClient(path=corpus_path)
        collections = client.list_collections()
        if not collections:
            return "Nessun corpus interno disponibile."

        oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        emb = oai.embeddings.create(
            model="text-embedding-3-small", input=query
        ).data[0].embedding

        col = collections[0]
        res = col.query(query_embeddings=[emb], n_results=3)
        docs = res["documents"][0] if res["documents"] else []
        return "\n\n".join(docs) if docs else "Nessun risultato nel corpus."
    except ImportError:
        return "chromadb non installato — corpus interno non disponibile."
    except Exception as e:
        return f"Errore nella ricerca nel corpus: {e}"


# Definizioni per il LLM (formato OpenAI tool use)
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "buscar_web",
            "description": "Cerca sul web usando DuckDuckGo. Restituisce una lista di risultati con titolo, URL e descrizione.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Termini di ricerca in inglese o italiano"},
                    "max_resultados": {"type": "integer", "description": "Numero massimo di risultati (default 5)", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "leer_url",
            "description": "Scarica ed estrae il testo di una URL. Usala per leggere il contenuto di una pagina web trovata con buscar_web.",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "URL completa da leggere"},
                    "max_chars": {"type": "integer", "description": "Massimo di caratteri da estrarre (default 4000)", "default": 4000},
                },
                "required": ["url"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_en_corpus",
            "description": "Cerca nel corpus interno di documenti indicizzati localmente (se esiste).",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Query di ricerca"},
                },
                "required": ["query"],
            },
        },
    },
]
'''

(AGENT_DIR / "__init__.py").write_text("")
(AGENT_DIR / "tools.py").write_text(TOOLS_PY.strip())
print("✓ agent/tools.py generato")

In [ ]:
# Test rapido degli strumenti
import sys
sys.path.insert(0, str(PROJECT_DIR))

from agent.tools import buscar_web, leer_url

print("=== buscar_web ===")
resultados = buscar_web("generative AI aquaculture Europe", max_resultados=3)
for r in resultados:
    print(f"  · {r['titulo'][:70]}")
    print(f"    {r['url']}")

print()
if resultados and resultados[0]["url"]:
    print("=== leer_url (prima URL) ===")
    texto = leer_url(resultados[0]["url"], max_chars=500)
    print(f"  {texto[:300]}...")

## Cella 2 — Pianificatore (`planner.py`)

Il pianificatore converte un tema in una lista di sottotask di ricerca.

In [ ]:
PLANNER_PY = '''
"""Crea un piano di ricerca a partire da un tema."""
import json
from openai import OpenAI
import os

MODELO_PLAN = "gpt-4o-mini"

PROMPT_PLANIFICACION = """Sei un ricercatore esperto di scouting tecnologico.
Dato il tema di ricerca seguente, crea un piano di ricerca strutturato
con 3-5 sottotask concreti e gestibili.

Tema: {tema}

Rispondi SOLO con JSON valido, senza testo aggiuntivo:
{{
  "subtareas": [
    {{"id": 1, "descripcion": "...", "query_sugerida": "..."}},
    ...
  ]
}}

Ogni sottotask deve essere:
- Specifico e attuabile (si può fare con ricerca web o lettura di URL)
- Diverso dagli altri (copre un angolo diverso del tema)
- Con una query_sugerida in inglese per la ricerca web"""


def crear_plan(tema: str) -> list[dict]:
    """
    Chiama il LLM per creare un piano di ricerca.
    Restituisce una lista di sottotask: [{id, descripcion, query_sugerida}].
    """
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model=MODELO_PLAN,
        messages=[{"role": "user", "content": PROMPT_PLANIFICACION.format(tema=tema)}],
        response_format={"type": "json_object"},
        temperature=0.3,
    )
    data = json.loads(resp.choices[0].message.content)
    return data.get("subtareas", [])
'''

(AGENT_DIR / "planner.py").write_text(PLANNER_PY.strip())
print("✓ agent/planner.py generato")

# Test
from agent.planner import crear_plan

TEMA_PRUEBA = "Applicazioni di intelligenza artificiale generativa nell'acquacoltura e pesca in Europa"
print(f"\nTema: {TEMA_PRUEBA}\n")
plan = crear_plan(TEMA_PRUEBA)

print(f"Piano generato ({len(plan)} sottotask):")
for s in plan:
    print(f"  {s['id']}. {s['descripcion']}")
    print(f"     Query: {s['query_sugerida']}")

## Cella 3 — Executor: ciclo ReAct per sottotask (`executor.py`)

Ogni sottotask del piano viene eseguito con un ciclo ReAct che può chiamare gli strumenti
fino a `MAX_PASOS` volte. Al termine, restituisce i frammenti raccolti.

In [ ]:
EXECUTOR_PY = '''
"""Esegue un sottotask del piano usando il ciclo ReAct + tool use."""
import json
import os
from openai import OpenAI
from agent.tools import buscar_web, leer_url, buscar_en_corpus, TOOL_DEFINITIONS

MODELO_EXEC  = "gpt-4o-mini"
MAX_PASOS    = 6

SYSTEM_EXECUTOR = """Sei un ricercatore autonomo. Il tuo compito è raccogliere informazioni
su un tema specifico usando gli strumenti disponibili.

Strategia:
1. Usa buscar_web per trovare fonti rilevanti
2. Usa leer_url per leggere il contenuto delle URL più promettenti
3. Usa buscar_en_corpus per consultare documenti interni se rilevante
4. Quando hai informazioni sufficienti, rispondi con un riassunto conciso
   della scoperta principale (3-5 frasi) e le URL usate.

Sii efficiente: non leggere più di 3 URL per sottotask."""


def _ejecutar_herramienta(nombre: str, args: dict) -> str:
    """Instrada la chiamata allo strumento corretto."""
    if nombre == "buscar_web":
        resultados = buscar_web(**args)
        return json.dumps(resultados, ensure_ascii=False)
    elif nombre == "leer_url":
        return leer_url(**args)
    elif nombre == "buscar_en_corpus":
        return buscar_en_corpus(**args)
    else:
        return f"Strumento sconosciuto: {nombre}"


def ejecutar_subtarea(subtarea: dict) -> dict:
    """
    Esegue un sottotask con il ciclo ReAct.
    Restituisce {subtarea_id, descripcion, hallazgo, fuentes, n_pasos}.
    """
    client   = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    messages = [
        {"role": "system",  "content": SYSTEM_EXECUTOR},
        {"role": "user",    "content": f"Investiga: {subtarea['descripcion']}\n"
                                       f"Query sugerida: {subtarea.get('query_sugerida', '')}"},
    ]
    fuentes_usadas = []
    n_pasos        = 0

    for _ in range(MAX_PASOS):
        n_pasos += 1
        resp = client.chat.completions.create(
            model=MODELO_EXEC,
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice="auto",
        )
        msg = resp.choices[0].message
        messages.append(msg)

        # Nessuna tool call → l'agente ha terminato
        if not msg.tool_calls:
            break

        # Esegue ogni tool call
        for tc in msg.tool_calls:
            nombre = tc.function.name
            args   = json.loads(tc.function.arguments)

            try:
                resultado = _ejecutar_herramienta(nombre, args)
            except Exception as e:
                resultado = f"Errore in {nombre}: {e}"

            if nombre == "leer_url" and args.get("url"):
                fuentes_usadas.append(args["url"])

            messages.append({
                "role":         "tool",
                "tool_call_id": tc.id,
                "content":      resultado,
            })

    hallazgo = msg.content or "Nessuna scoperta concreta."
    return {
        "subtarea_id":  subtarea["id"],
        "descripcion":  subtarea["descripcion"],
        "hallazgo":     hallazgo,
        "fuentes":      list(dict.fromkeys(fuentes_usadas)),  # deduplica
        "n_pasos":      n_pasos,
    }
'''

(AGENT_DIR / "executor.py").write_text(EXECUTOR_PY.strip())
print("✓ agent/executor.py generato")

# Test con il primo sottotask del piano generato nella cella precedente
from agent.executor import ejecutar_subtarea

if plan:
    print(f"\nEsecuzione sottotask 1: {plan[0]['descripcion']}\n")
    resultado = ejecutar_subtarea(plan[0])

    print(f"Scoperta ({resultado['n_pasos']} passi):")
    print(resultado['hallazgo'][:400])
    print(f"\nFonti usate: {resultado['fuentes']}")

## Cella 4 — Sintetizzatore (`synthesizer.py`)

Riceve tutte le scoperte del piano e genera il report finale in Markdown.

In [ ]:
SYNTHESIZER_PY = '''
"""Genera il report finale a partire dalle scoperte raccolte."""
import os
from openai import OpenAI

MODELO_SINTESIS = "gpt-4o-mini"

PROMPT_INFORME = """Sei un analista senior di scouting tecnologico.
A partire dalle seguenti scoperte di ricerca, genera un report
professionale e strutturato sul tema.

TEMA: {tema}

SCOPERTE:
{hallazgos}

ISTRUZIONI:
- Scrivi in italiano, tono analitico e diretto
- Struttura: Riepilogo esecutivo (3-5 righe) → Scoperte per area → Conclusioni → Fonti
- Sintetizza e interpreta, non copiare letteralmente
- Segnala tendenze chiave, attori rilevanti e gap informativi
- Cita le fonti con [URL] alla fine di ogni affermazione che ne ha bisogno
- Formato Markdown pulito e leggibile"""


def generar_informe(tema: str, resultados: list[dict]) -> str:
    """
    Genera il report finale.
    resultados: lista di dict {subtarea_id, descripcion, hallazgo, fuentes}.
    Restituisce il report come stringa Markdown.
    """
    # Formatta le scoperte per il prompt
    partes = []
    for r in resultados:
        fuentes_str = "\n".join(f"  - {f}" for f in r.get("fuentes", []))
        partes.append(
            f"### Sottotask {r['subtarea_id']}: {r['descripcion']}\n"
            f"{r['hallazgo']}\n"
            f"Fonti:\n{fuentes_str or '  (nessuna fonte esterna)'}"
        )
    hallazgos_txt = "\n\n".join(partes)

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp   = client.chat.completions.create(
        model=MODELO_SINTESIS,
        messages=[{
            "role":    "user",
            "content": PROMPT_INFORME.format(tema=tema, hallazgos=hallazgos_txt),
        }],
        temperature=0.4,
        max_tokens=2000,
    )
    return resp.choices[0].message.content
'''

(AGENT_DIR / "synthesizer.py").write_text(SYNTHESIZER_PY.strip())
print("✓ agent/synthesizer.py generato")

## Cella 5 — Classe principale `ResearchAgent` (`agent.py`)

Orchestra il flusso completo: pianificare → eseguire ogni sottotask → sintetizzare → salvare.

In [ ]:
AGENT_PY = '''
"""ResearchAgent: agente autonomo di ricerca e scouting tecnologico."""
import json
import time
from dataclasses import dataclass, field
from pathlib import Path
from datetime import datetime

from agent.planner    import crear_plan
from agent.executor   import ejecutar_subtarea
from agent.synthesizer import generar_informe


@dataclass
class ResultadoInvestigacion:
    tema:           str
    plan:           list[dict]
    resultados:     list[dict]
    informe:        str
    duracion_s:     float
    n_fuentes:      int
    ruta_informe:   str = ""


class ResearchAgent:
    """
    Agente autonomo di ricerca.

    Uso:
        agent = ResearchAgent(informes_dir="./informes")
        resultado = agent.investigar("IA generativa nell'acquacoltura")
        print(resultado.informe)
    """

    def __init__(
        self,
        informes_dir: str = "./informes",
        verbose: bool = True,
    ):
        self.informes_dir = Path(informes_dir)
        self.informes_dir.mkdir(parents=True, exist_ok=True)
        self.verbose = verbose

    def _log(self, msg: str):
        if self.verbose:
            print(msg)

    def investigar(self, tema: str) -> ResultadoInvestigacion:
        """
        Esegue il ciclo completo di ricerca.
        1. Pianifica i sottotask
        2. Esegue ogni sottotask (ReAct + tools)
        3. Sintetizza il report finale
        4. Salva il report su disco
        """
        inicio = time.time()
        self._log(f"\n🔍 Avvio ricerca: {tema}\n")

        # Fase 1: Pianificazione
        self._log("📋 Fase 1/3 — Pianificazione...")
        plan = crear_plan(tema)
        self._log(f"   Piano: {len(plan)} sottotask")
        for s in plan:
            self._log(f"   {s['id']}. {s['descripcion']}")

        # Fase 2: Esecuzione dei sottotask
        self._log("\n🔎 Fase 2/3 — Ricerca in corso...")
        resultados = []
        for subtarea in plan:
            self._log(f"   Sottotask {subtarea['id']}: {subtarea['descripcion'][:60]}...")
            resultado = ejecutar_subtarea(subtarea)
            resultados.append(resultado)
            self._log(f"   → {resultado['n_pasos']} passi, {len(resultado['fuentes'])} fonti")

        # Fase 3: Sintesi
        self._log("\n✍️  Fase 3/3 — Sintesi del report...")
        informe = generar_informe(tema, resultados)

        duracion_s = time.time() - inicio
        n_fuentes  = sum(len(r["fuentes"]) for r in resultados)

        # Salva il report
        fecha         = datetime.now().strftime("%Y%m%d_%H%M")
        nombre_slug   = tema[:40].replace(" ", "_").replace("/", "-")
        ruta_informe  = self.informes_dir / f"{fecha}_{nombre_slug}.md"

        cabecera = (
            f"---\n"
            f"tema: {tema}\n"
            f"fecha: {datetime.now().isoformat()}\n"
            f"n_subtareas: {len(plan)}\n"
            f"n_fuentes: {n_fuentes}\n"
            f"duracion_s: {duracion_s:.1f}\n"
            f"---\n\n"
        )
        ruta_informe.write_text(cabecera + informe, encoding="utf-8")

        self._log(f"\n✅ Report salvato: {ruta_informe}")
        self._log(f"   Durata: {duracion_s:.1f}s | Fonti: {n_fuentes}")

        return ResultadoInvestigacion(
            tema=tema,
            plan=plan,
            resultados=resultados,
            informe=informe,
            duracion_s=duracion_s,
            n_fuentes=n_fuentes,
            ruta_informe=str(ruta_informe),
        )
'''

(AGENT_DIR / "agent.py").write_text(AGENT_PY.strip())
print("✓ agent/agent.py generato")

## Cella 6 — Test dell'agente completo

Esegue il ciclo completo: piano → ricerca → report.

In [ ]:
import importlib
import agent.agent
importlib.reload(agent.agent)

from agent.agent import ResearchAgent

agente = ResearchAgent(
    informes_dir=str(INFORMES_DIR),
    verbose=True,
)

# Tema di esempio direttamente rilevante per DIH DATAlife
TEMA = "Applicazioni di intelligenza artificiale nell'industria alimentare galiziana"

resultado = agente.investigar(TEMA)

In [ ]:
# Mostra il report generato
from IPython.display import Markdown, display

display(Markdown(resultado.informe))

print(f"\n--- Metadati ---")
print(f"Sottotask eseguiti:   {len(resultado.plan)}")
print(f"Fonti consultate:     {resultado.n_fuentes}")
print(f"Durata:               {resultado.duracion_s:.1f}s")
print(f"Report salvato in:    {resultado.ruta_informe}")

## Cella 7 — CLI (`app.py`)

In [ ]:
APP_PY = '''
#!/usr/bin/env python3
"""
ResearchAgent CLI.
Uso: python app.py "Tema di ricerca"
     python app.py "IA nell'acquacoltura" --informes-dir ./miei-report
"""
import sys
import argparse
from dotenv import load_dotenv

load_dotenv()

from agent.agent import ResearchAgent


def main():
    parser = argparse.ArgumentParser(description="Agente di ricerca autonomo")
    parser.add_argument("tema", help="Tema da investigare")
    parser.add_argument(
        "--informes-dir", default="./informes",
        help="Directory dove salvare i report (default: ./informes)"
    )
    parser.add_argument(
        "--quiet", action="store_true",
        help="Sopprime i messaggi di avanzamento"
    )
    args = parser.parse_args()

    agente = ResearchAgent(
        informes_dir=args.informes_dir,
        verbose=not args.quiet,
    )
    resultado = agente.investigar(args.tema)

    print(f"\nReport: {resultado.ruta_informe}")
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

REQUIREMENTS = """openai>=1.0.0
httpx>=0.27.0
beautifulsoup4>=4.12.0
python-dotenv>=1.0.0
chromadb>=0.5.0
"""

README = """# ResearchAgent — Agente di Ricerca Autonomo

Agente che riceve un tema, pianifica la sua ricerca, cerca fonti sul web,
legge e riassume il contenuto, e consegna un report strutturato in Markdown.

## Installazione

```bash
cd proyectos/research-agent
pip install -r requirements.txt
```

Crea un `.env`:
```
OPENAI_API_KEY=sk-...
```

## Uso

```bash
python app.py "Applicazioni di IA nell'acquacoltura in Europa"
python app.py "Tendenze nella bioeconomia forestale" --informes-dir ./miei-report
```

I report vengono salvati in `informes/YYYYMMDD_HHMM_Tema.md`.

## Architettura

```
agent/
├── tools.py        # buscar_web, leer_url, buscar_en_corpus
├── planner.py      # crear_plan() con LLM (JSON mode)
├── executor.py     # ciclo ReAct per sottotask
├── synthesizer.py  # generar_informe() a partire dalle scoperte
└── agent.py        # ResearchAgent: orchestra il flusso completo
```

## Costo approssimativo

Con `gpt-4o-mini` e 4 sottotask:
- Per report: ~$0.005–0.02
- 100 report/mese: ~$0.50–2.00

## Progetto 2 — Piano di Formazione AI Engineering (Settimana 15)
"""

(PROJECT_DIR / "app.py").write_text(APP_PY.strip())
(PROJECT_DIR / "requirements.txt").write_text(REQUIREMENTS)
(PROJECT_DIR / "README.md").write_text(README)

print("✓ app.py generato")
print("✓ requirements.txt generato")
print("✓ README.md generato")
print()
print("Struttura del progetto:")
import subprocess
subprocess.run(["find", str(PROJECT_DIR), "-type", "f", "-not", "-path", "*/__pycache__/*"],
               capture_output=False)

## Sfida — Aggiungere il replanning quando un sottotask fallisce

L'executor attuale registra l'errore e restituisce una scoperta vuota. Un agente robusto dovrebbe
ripianificare: se un sottotask non produce risultati utili, generare un sottotask alternativo.

**Implementa** `replanificar_subtarea()` in `planner.py`:

```python
def replanificar_subtarea(tema: str, subtarea_fallida: dict, razon: str) -> dict:
    """
    Genera un sottotask alternativo quando l'originale non ha prodotto risultati.

    Parametri:
    - tema: il tema generale di ricerca
    - subtarea_fallida: il sottotask che non ha funzionato {id, descripcion, query_sugerida}
    - razon: perché è fallito (es. "nessun risultato su DuckDuckGo", "URL non accessibile")

    Restituisce: nuovo sottotask con query_sugerida diversa
    """
    # TODO: chiama il LLM con JSON mode per generare un sottotask alternativo
    # Il prompt deve includere il sottotask fallito e la ragione del fallimento
    # affinché il LLM non ripeta lo stesso errore
    pass
```

Poi modifica `ResearchAgent.investigar()` affinché se `resultado['hallazgo']` contiene
"Error" o ha meno di 50 caratteri, chiami `replanificar_subtarea()` e riprovi una volta.

**Suggerimento:** definisci una soglia: `if len(resultado['hallazgo']) < 100 and resultado['n_pasos'] == 1`

In [ ]:
# Scheletro della sfida
RETO_SNIPPET = '''
# In planner.py — aggiungi:

PROMPT_REPLANNING = """Il seguente sottotask di ricerca non ha prodotto risultati utili.

Tema generale: {tema}
Sottotask fallito: {descripcion}
Query provata: {query_sugerida}
Ragione del fallimento: {razon}

Genera UN sottotask alternativo che copra lo stesso angolo ma con una strategia diversa.
Rispondi SOLO con JSON:
{{"id": {id}, "descripcion": "...", "query_sugerida": "..."}}"""


def replanificar_subtarea(tema: str, subtarea_fallida: dict, razon: str) -> dict:
    # TODO: implementare
    pass


# In agent.py — modifica il ciclo di esecuzione:

for subtarea in plan:
    resultado = ejecutar_subtarea(subtarea)

    # TODO: se la scoperta è molto corta o contiene "Error",
    # chiama replanificar_subtarea() e riprova

    resultados.append(resultado)
'''
print(RETO_SNIPPET)

## Riepilogo della settimana 15

| Componente | File | Pattern riutilizzato |
|-----------|---------|-------------------|
| Strumenti | `agent/tools.py` | Tool use (S09) |
| Pianificatore | `agent/planner.py` | Plan-and-Execute (S11) + JSON mode (S05) |
| Executor | `agent/executor.py` | Ciclo ReAct (S09) |
| Sintetizzatore | `agent/synthesizer.py` | Prompt engineering (S02) |
| Orchestratore | `agent/agent.py` | Supervisor pattern (S11) |
| CLI | `app.py` | — |
| Replanning | Sfida in `planner.py` | Replanning (S11) |

La settimana 16 aggiunge **API REST + test** a ResearchAgent, replicando il pattern della settimana 13
ma con gli endpoint propri di un agente asincrono di lunga durata.